In [ ]:
# Colab setup: mount Google Drive and configure project paths.
# This notebook is Colab-first (consistent with the other 4 notebooks)
# but also works locally if you happen to have everything checked out.
import os
from pathlib import Path

# --- Try Colab first ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
    PROJECT_DIR = "/content/drive/MyDrive/Aini/ml-assignment/Team-Assignment-2/binus-ai-2026sem3-assignment2-group04"
    MODEL_DIR   = f"{PROJECT_DIR}/models"
    FIGURES_DIR = f"{PROJECT_DIR}/report/figures"
    RESULTS_CSV = f"{PROJECT_DIR}/report/realworld_results.csv"
except ImportError:
    ON_COLAB = False
    cwd = Path.cwd()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
    os.chdir(PROJECT_ROOT)
    MODEL_DIR   = "models"
    FIGURES_DIR = "report/figures"
    RESULTS_CSV = "report/realworld_results.csv"

MODEL_PATH = f"{MODEL_DIR}/transfer_mobilenet_v1.keras"

# --- Locate test_images/ ---
# Priority: directly uploaded to /content (Colab), then Drive, then cloned repo, then local.
candidates = [
    "/content/test_images",                               # drag-and-dropped to Colab Files panel
    f"{PROJECT_DIR}/test_images" if ON_COLAB else None,   # uploaded to Drive
    "/content/repo/test_images",                          # cloned from GitHub
    "test_images",                                        # local
]
TEST_IMAGES_DIR = None
for c in candidates:
    if c and Path(c).is_dir() and any(Path(c).iterdir()):
        TEST_IMAGES_DIR = c
        break

if TEST_IMAGES_DIR is None:
    raise RuntimeError(
        "test_images/ not found. On Colab, drag the entire test_images "
        "folder into the file panel on the left (the Files icon), or run:\n"
        "  !git clone https://github.com/ainichan22/binus-ai-2026sem3-assignment2-group04.git /content/repo"
    )

import tensorflow as tf
print("On Colab:    ", ON_COLAB)
print("TensorFlow:  ", tf.__version__)
print("GPU:         ", tf.config.list_physical_devices("GPU") or "(none — CPU is fine for 30 images)")
print("Model path:  ", MODEL_PATH)
print("Test images: ", TEST_IMAGES_DIR)
print("Figures out: ", FIGURES_DIR)

# Group Assignment 2 — Phase 3A: Real-World Image Testing

**Objective:**
Evaluate the trained transfer-learning model (`transfer_mobilenet_v1.keras`, test accuracy 0.8993 on the CIFAR-10 test set) on **30 external images** sourced from the open web. The CIFAR-10 test split is in-distribution by construction; this notebook quantifies how the model behaves on out-of-distribution real-world photos and identifies systematic failure modes for the report's ch.8 discussion.

**Methodology:**
- 30 images, 3 per class, organized as `test_images/<class>/<class>_NN.jpg`.
- Sources: Unsplash, Pexels, free Google Images results. Excluded: CIFAR-10 originals and AI-generated images.
- The same preprocessing pipeline as inference (`center_crop_square → resize 96 → mobilenet_v2.preprocess_input`) so this notebook mirrors what the deployed Streamlit app does.

**How to run on Colab:**
1. Open this notebook in Colab.
2. Run cell 1 — it mounts Drive and asks where the test images live.
3. **Drag the entire `test_images` folder** from your local repo into Colab's file panel (the Files icon on the left sidebar). After upload completes, you should see `/content/test_images/` populated.
4. Restart and run all.

**Expected finding:**
Real-world accuracy will be visibly lower than the 0.8993 CIFAR-test number. The gap quantifies the domain shift between 32×32 thumbnails and natural high-resolution photos, which is the central limitation flagged throughout the report.

### Section 0: Setup and Reproducibility

This notebook does inference only — no training, no randomness in modeling. Results are bit-deterministic given the same input images and the same `transfer_mobilenet_v1.keras` artifact. CPU inference is fine; running the notebook end-to-end takes under a minute even on a free Colab CPU.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Class names — must match the order the model was trained on
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = 96    # MobileNetV2 input size we trained at

print(f"Classes: {CLASS_NAMES}")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")

### Section 1: Locate the Real-World Test Set

Walk `test_images/<class>/*.jpg` and assemble a list of `(image_path, true_label_idx)` pairs. We expect 30 images — exactly 3 per class — and verify that count before proceeding.

In [ ]:
def collect_test_images(root: str):
    """Returns a list of (path, true_label_idx) tuples."""
    items = []
    for label_idx, class_name in enumerate(CLASS_NAMES):
        class_dir = Path(root) / class_name
        if not class_dir.is_dir():
            print(f"  ✗ missing directory: {class_dir}")
            continue
        for img_path in sorted(class_dir.glob("*.jpg")):
            items.append((str(img_path), label_idx))
    return items

test_items = collect_test_images(TEST_IMAGES_DIR)
print(f"Found {len(test_items)} images total.")
for class_idx, class_name in enumerate(CLASS_NAMES):
    n = sum(1 for _, idx in test_items if idx == class_idx)
    print(f"  {class_name:12s}: {n}")

assert len(test_items) >= 30, f"Expected at least 30 images, got {len(test_items)}"

### Section 2: Load the Transfer-Learning Model

The model artifact must already exist on Drive at `models/transfer_mobilenet_v1.keras` (it was produced by `03_transfer_learning_EN.ipynb` Section 12 and downloaded back here in Phase 1B).

In [ ]:
assert os.path.exists(MODEL_PATH), (
    f"Model not found at {MODEL_PATH}.\n"
    "Make sure 03_transfer_learning_EN.ipynb has been run end-to-end so the "
    "transfer_mobilenet_v1.keras artifact is in your Drive's models/ folder."
)

print(f"Loading model from {MODEL_PATH} ...")
model = tf.keras.models.load_model(MODEL_PATH)
print(f"Loaded. Input shape: {model.input_shape}, output: {model.output_shape}")
# Warm-up: pay graph-tracing cost once so the per-image timing below is stable
_ = model(np.zeros((1, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32), training=False)
print("Model ready.")

### Section 3: Preprocessing Pipeline

Identical to `app/utils/preprocess.py` (which itself mirrors the training notebook). Three steps:

1. **`center_crop_square`** — crop the largest centered square from a possibly-rectangular photo. CIFAR's training images are square; uncropped real-world photos would be distorted by a naive resize.
2. **Resize to 96×96** with bilinear interpolation.
3. **`preprocess_input`** — map [0, 255] → [-1, 1], matching MobileNetV2's ImageNet pretraining distribution.

In [ ]:
def center_crop_square(arr: np.ndarray) -> np.ndarray:
    h, w = arr.shape[:2]
    s = min(h, w)
    y, x = (h - s) // 2, (w - s) // 2
    return arr[y:y + s, x:x + s]

def load_and_preprocess(path: str) -> tuple:
    """Returns (display_rgb_uint8, model_input_float32) for a single image."""
    img = Image.open(path).convert("RGB")
    rgb = np.array(img)                               # (H, W, 3) uint8
    rgb_cropped = center_crop_square(rgb)
    x = tf.cast(rgb_cropped, tf.float32)
    x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE))
    x = preprocess_input(x)
    return rgb_cropped, x.numpy()

# Sanity check
sample_path = test_items[0][0]
disp, model_in = load_and_preprocess(sample_path)
print(f"Sample: {sample_path}")
print(f"  display shape (after center-crop): {disp.shape}")
print(f"  model input shape:                 {model_in.shape}")
print(f"  pixel range:                       [{model_in.min():.3f}, {model_in.max():.3f}]  (target: [-1, 1])")

### Section 4: Batch Prediction

For each image: preprocess, run a single forward pass via `model(x)` (faster than `model.predict()` for one-image batches), record the Top-3 prediction indices, the Top-1 confidence, and whether the Top-1 matches the true label. Store everything in a DataFrame for downstream analysis.

In [ ]:
rows = []
for path, true_idx in test_items:
    _, x = load_and_preprocess(path)
    proba = model(x[np.newaxis], training=False).numpy()[0]
    top3_idx = np.argsort(proba)[-3:][::-1]
    rows.append({
        "filename":     Path(path).name,
        "true_class":   CLASS_NAMES[true_idx],
        "true_idx":     int(true_idx),
        "pred_class":   CLASS_NAMES[int(top3_idx[0])],
        "pred_idx":     int(top3_idx[0]),
        "confidence":   float(proba[top3_idx[0]]),
        "top3":         "→".join(CLASS_NAMES[int(i)] for i in top3_idx),
        "top3_conf":    "→".join(f"{proba[int(i)]:.2f}" for i in top3_idx),
        "correct":      bool(top3_idx[0] == true_idx),
    })

df = pd.DataFrame(rows)
print(f"Predicted {len(df)} images.")
df

### Section 5: Aggregate and Per-Class Accuracy

The headline number for the report. We compare the real-world accuracy directly against the CIFAR-test number (0.8993) the model achieved on the in-distribution split.

In [ ]:
CIFAR_TEST_ACC = 0.8993       # from notebooks/03_transfer_learning_EN.ipynb Section 8

overall_acc = df["correct"].mean()
print(f"Real-world accuracy:  {overall_acc:.4f}  ({df['correct'].sum()}/{len(df)} correct)")
print(f"CIFAR test-set acc:   {CIFAR_TEST_ACC:.4f}")
print(f"Domain gap:           {(CIFAR_TEST_ACC - overall_acc) * 100:+.2f} pp\n")

per_class = (
    df.groupby("true_class")["correct"]
      .agg(["mean", "sum", "count"])
      .rename(columns={"mean": "accuracy", "sum": "n_correct", "count": "n_total"})
      .reindex(CLASS_NAMES)
)
print("Per-class accuracy (real-world):")
print(per_class.to_string())

### Section 6: Failure-Case Visualization

Build a grid showing every misclassified image, annotated with the true class and the model's top prediction (with confidence). This is the diagnostic figure for the report — looking at the actual failure images is the only way to identify what *kind* of domain shift is hurting the model.

In [ ]:
failures = df[~df["correct"]].reset_index(drop=True)
print(f"Failures: {len(failures)} / {len(df)} ({len(failures) / len(df):.1%})")
print()

if len(failures) == 0:
    print("Perfect score — no failure grid to plot.")
else:
    n = len(failures)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for i, row in failures.iterrows():
        path = Path(TEST_IMAGES_DIR) / row["true_class"] / row["filename"]
        img, _ = load_and_preprocess(str(path))
        axes[i].imshow(img)
        axes[i].set_title(
            f"true: {row['true_class']}\npred: {row['pred_class']} ({row['confidence']:.0%})",
            fontsize=10,
        )

    plt.suptitle(
        f"Real-world failures — {len(failures)} of {len(df)} ({len(failures) / len(df):.1%})",
        fontsize=13, y=1.0,
    )
    plt.tight_layout()
    plt.show()

### Section 7: Discussion

**What this measures.** The CIFAR-10 test set is bit-perfect in-distribution: same 32×32 source, same camera-style thumbnails, same dataset author curation. The 30 real-world images here are deliberately *out* of that distribution — high-resolution natural photos sourced from public photo libraries, then center-cropped and resized to 96×96 by our pipeline. The accuracy gap between the CIFAR-test number and the real-world number is therefore a measurement of domain shift rather than fundamental model weakness.

**Failure modes to look for in the grid above.** Three patterns recur in CIFAR-trained models on natural photos: (1) **scale mismatch** — real-world objects can fill the frame or sit small in a complex scene, while CIFAR objects always occupy most of the 32×32 frame; (2) **background dominance** — natural backgrounds (foliage, sky, water) carry richer texture than CIFAR thumbnails, sometimes pulling predictions toward classes whose training distribution shares the background; (3) **class-pair confusion** — the same animal-pair confusions visible in the CIFAR confusion matrix (cat/dog, deer/horse, bird/airplane) re-appear here, often more pronounced.

**Implication for the deployed app.** The Streamlit Predict page already includes a disclaimer about the domain gap. Users uploading high-resolution natural photos should expect lower confidence and occasional misclassifications, especially on the difficult animal-pair classes. Stronger augmentation or a larger backbone could close some of this gap, but eliminating it entirely would require fine-tuning on labeled real-world images — work flagged as "future work" in the report's ch.9.

### Section 8: Save Artifacts for the Report

Persist three outputs that the report's ch.8 will reference:

- **`report/realworld_results.csv`** — full per-image table.
- **`report/figures/realworld-failures.png`** — annotated failure grid.
- **`report/figures/realworld-per-class.png`** — per-class accuracy bar chart.

On Colab, all three are written to your Drive (`PROJECT_DIR/report/...`). Download them back to your local repo's `report/` folder before committing.

In [ ]:
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)

# 1) CSV with full per-image results
df.to_csv(RESULTS_CSV, index=False)
print(f"Saved {RESULTS_CSV}  ({os.path.getsize(RESULTS_CSV) / 1024:.1f} KB, {len(df)} rows)")

# 2) Failure grid as standalone PNG (the inline plt.show() above doesn't save)
if len(failures) > 0:
    n = len(failures); cols = 4; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    axes = np.array(axes).reshape(-1)
    for ax in axes: ax.axis("off")
    for i, row in failures.iterrows():
        path = Path(TEST_IMAGES_DIR) / row["true_class"] / row["filename"]
        img, _ = load_and_preprocess(str(path))
        axes[i].imshow(img)
        axes[i].set_title(
            f"true: {row['true_class']}\npred: {row['pred_class']} ({row['confidence']:.0%})",
            fontsize=10,
        )
    plt.suptitle(
        f"Real-world failures — {len(failures)} of {len(df)} ({len(failures) / len(df):.1%})",
        fontsize=13, y=1.0,
    )
    plt.tight_layout()
    out_path = f"{FIGURES_DIR}/realworld-failures.png"
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"Saved {out_path}  ({os.path.getsize(out_path) / 1024:.1f} KB)")
else:
    print("No failures — no failure grid saved.")

# 3) Per-class accuracy bar chart
fig, ax = plt.subplots(figsize=(10, 4))
acc_vals = per_class["accuracy"].values
bars = ax.bar(per_class.index, acc_vals, color="steelblue", edgecolor="black")
ax.axhline(CIFAR_TEST_ACC, color="red", linestyle="--", label=f"CIFAR test acc = {CIFAR_TEST_ACC:.3f}")
ax.set_ylabel("Real-world accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Per-class accuracy on 30 real-world images")
for bar, v in zip(bars, acc_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f"{v:.2f}",
            ha="center", fontsize=9)
ax.legend()
plt.tight_layout()
bar_path = f"{FIGURES_DIR}/realworld-per-class.png"
plt.savefig(bar_path, dpi=120, bbox_inches="tight")
plt.close()
print(f"Saved {bar_path}  ({os.path.getsize(bar_path) / 1024:.1f} KB)")